# **Finetuning to Follow Instructions**

**This chapter covers**

- Introduction to the instruction finetuning process of LLMs
- Preparing a dataset for supervised instruction finetuning
- Organizing instruction data in training batches
- Loading a pretrained LLM and finetuning it to follow human instructions
- Extracting LLM-generated instruction responses for evaluation
- Evaluating an instruction-finetuned LLM

![Alt text](../../assests/figure71.png)

A mental model of the three main stages of coding an LLM, pretraining the LLM on a general text dataset, and finetuning it. This section focuses on finetuning a pretrained LLM to follow human instructions.

In [1]:
from importlib.metadata import version

pkgs = [
    "numpy",       # PyTorch & TensorFlow dependency
    "matplotlib",  # Plotting library
    "tiktoken",    # Tokenizer
    "torch",       # Deep learning library
    "tqdm",        # Progress bar
    "tensorflow",  # For OpenAI's pretrained weights
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

numpy version: 1.26.4
matplotlib version: 3.10.6
tiktoken version: 0.11.0
torch version: 2.4.0
tqdm version: 4.67.1
tensorflow version: 2.20.0


### **1. Introduction to instruction finetuning**

- In chapter 5, we saw that pretraining an LLM involves a training procedure where it learns to generate one word at a time

- Hence, a pretrained LLM is good at text completion, but it is not good at following instructions, such as "Fix the grammar in this text" or "Convert this text into passive voice"

- This chapter focuses on improving the LLM's ability to follow such instructions and generate a desired respones.


![Alt text](../../assests/figure72.png)

This figure shows examples of instructions that are processed by an LLM to generate desired responses.


- The topics covered in this chapter are summarized in the figure below

![Alt text](../../assests/figure73.png)

The figure above shows the three-stage process for instruction finetuning the LLM in this chapter.
- stage 1: involves dataset preparation.
- stage 2: focuses on model setup and finetuning.
- stage 3: covers the evaluation of the model.

### **2. Preparing a dataset for supervised instruction finetuning**

- In this section, we download and format the instruction dataset for instruction finetuning a pretrained LLM.
- The dataset consists of 1100 instuction-response pairs.
- This dataset has been specifically created for this.

- The code below implements and executes a function to download this dataset, which is a relatively small file in JSON format.

In [10]:
import json
import os
import urllib


def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    # The book originally contained this unnecessary "else" clause:
    #else:
    #    with open(file_path, "r", encoding="utf-8") as file:
    #        text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


- Each item in the data list we loaded from the JSON file above is a dictionary in the following form

In [11]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


- Note that the `'input'` field can be empty:

In [12]:
print("Another example entry:\n", data[999])

Another example entry:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


In [13]:
data[:5]

[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'},
 {'instruction': 'Edit the following sentence for grammar.',
  'input': 'He go to the park every day.',
  'output': 'He goes to the park every day.'},
 {'instruction': 'Convert 45 kilometers to meters.',
  'input': '',
  'output': '45 kilometers is 45000 meters.'},
 {'instruction': "Rewrite this sentence to start with 'Although': Despite the rain, they went for a walk.",
  'input': '',
  'output': 'Although it was raining, they went for a walk.'},
 {'instruction': 'What are the first 10 square numbers?',
  'input': '',
  'output': '1, 4, 9, 16, 25, 36, 49, 64, 81, 100.'}]

- Instruction finetuning is often referred to as `"supervised instruction finetuning"` because it involves training a model on a dataset where the `input-output` pairs are explicitly provided

- There are different ways to format the entries as inputs to the LLM; the figure below illustrates two example formats that were used for training the Alpaca (https://crfm.stanford.edu/2023/03/13/alpaca.html) and Phi-3 (https://arxiv.org/abs/2404.14219) LLMs, respectively.

![Alt text](../../assests/figure74.png)

Figure above: Comparison of prompt styles for instruction finetuning in LLMs. The `Alpaca style (left)` uses a structured format with defined sections for `instruction`, `input`, and `response`, while the `Phi-3` style (right) employs a simpler format with designed `<|user|>` and `<|assistant|>` tokens.

- Let's define a `format_input` function that we can use to convert the entries in the `data` list into the `Alpaca-style` input format.

In [14]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- This `format_input` function takes a dictionary `entry` as input and constructs a formatted string. 
- Let's test it to dataset entry `data[50]`

In [15]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


- Below is a formatted response without an input field

In [16]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


- As seen based on the following output, entries with an empty `'input'` field don't contain an `### input:` section in the formatted input.

- Lastly, before we prepare the PyTorch data loaders in the next section, we divide the dataset into a `training`, `validation`, and `test` set.

- Here's how we calculate the portions:

In [17]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [18]:
print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### **3. Organizing data into training batches**

- The next step focuses on constructing the training batches effectively. 
- This involves defining a method that will ensure our model receives the formatted training data during the finetuning process.

![Alt text](../../assests/figure75.png)


- Batching process for instruction finetuning is a bit more involved and requires us to create our own custom collate function that we will later plug into the `DataLoader`.

- We implement this custom collate function to handle the specific requirements and formatting of our instruction finetuning dataset.

- We will tackle the `batching process` in several steps including the coding of the custom collate function.

![Alt text](../../assests/figure76.png)

An illustration of the five substeps involved in implementing the batching process: applying the prompt template defined in the previous section, using tokenization from previous chapters, adding padding tokens, creating target token IDs, and replacing -100 placeholder tokens to mask padding tokens in the loss function.

- First, we code an `InstructionDataset` class that applies `format_input` from the previous section and `pretokenizes` all inputs in the dataset, similar to the `SpamDataset` in previous chapter.

![Alt text](../../assests/figure77.png)



In [21]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- We aim to accelerate training by collecting multiple training examples in a batch, which necessitates `padding` all inputs to a similar length. 
- We use the `<|endoftext|>` token as a `padding token`.

- Instead of appending the `<|endoftext|>` tokens to the text inputs, we can append its token ID to the pre-tokenized inputs directly.
- To remind us which token ID we should use, we can use the tokenizer's `.encode` method on an `<|endoftext|>` token:

In [22]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


- In chapter 6, we padded all examples in a dataset to the same length

  - Here, we take a more sophisticated approach and develop a custom "collate" function that we can pass to the data loader
  
  - This custom collate function pads the training examples in each batch to have the same length (but different batches can have different lengths)


![Alt text](../../assests/figure78.png)


This figure showed the padding of training examples in batches using token ID `50256` to ensure uniform length within each batch. Each batch may have different lengths, as shown in the first and second batches in the figure above.

In [23]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch
    # and increase the max length by +1, which will add one extra
    # padding token below
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to batch_max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # Via padded[:-1], we remove the extra padded token
        # that has been added via the +1 setting in batch_max_length
        # (the extra padding token will be relevant in later codes)
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # Convert list of inputs to tensor and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

- The `custom_collate_draft_1` function is designed to be integrated into PyTorch `DataLoader`, but it can also function as a standalone tool.
- Here, we use it independently to test and verify that it operates as intended.
- Let's try it on three different inputs that we want to assemble into a batch, where each example gets padded to the same length:

In [39]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


- As we can see based on the preceding output, all inputs have been padded to the length of the longest input list, `inputs_1` containing 5 token IDs.

- So, we just implemented our first custom collate function to create batches from lists of inputs. 
  
- We also need to create batches with the target token IDs, corresponding to the batch of input IDs.
  
- These target IDs, are crucial because they represent what we want the model to generate and what we need during training to calculate the loss for the weight updates. 


![Alt text](../../assests/figure79.png)


- Above, we only returned the inputs to the LLM; however, for LLM training, we also need the target values

- Similar to pretraining an LLM, the targets are the inputs shifted by 1 position to the right, so the LLM learns to predict the next token


![Alt text](../../assests/figure710.png)

This figure illustrates the input and target token alignment used in the instruction finetuning process of an LLM. For each input sequence, the corresponding target sequence is created by shifting the token IDs one position to the right, omitting the first token of the input, and appending an end-of-text-token.

In [40]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs to tensor and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

- Applied to the example `batch` consisting of 3 input lists we defined earlier, the new `custom_collate_draft_2` function now returns the input and the target batch:

In [41]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


- Next, we introduce an `ignore_index` value to replace all padding token IDs with a new value; the purpose of this `ignore_index` is that we can ignore padding values in the loss function (more on that later)


![Alt text](../../assests/figure711.png)

- After creating the target sequence by shifting token IDs one position to the right and appending an end-of-text token, `step 2.5` focuses on replacing end-of-text padding tokens with a placeholder value `(-100)`

- We replace the end-of-text tokens, which we previously used as padding tokens and are assigned token ID `50256`, with `-100` in the target token list. 

- As shown in the figure below, note that we retain one end-of-text token, ID 50256, in the target list. This allows the LLM to learn when to generate and end-of-text token in response to instructions, which we use as indicator that the generated response is complete.


![Alt text](../../assests/figure712.png)

- Shows the replacement of all but the first instance of the end-of-text token, which we use as padding, with the placeholder value `-100`, while keeping the initial end-of-text token in each target sequence.

- In the following code, we modify our custom collate function to replace tokens with ID `50256` with `-100` in the target lists. 
- Additionally, we introduce an `allowed_max_length` parameter to optionally limit the length of the samples.
- This adjustment will be useful if you plan to work with your own datasets that exceed the 1024-token context size supported by the GPT-2 model.
- The code for this updated collate function is as follows:

In [42]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

- Again, let's try the collate function on the sample batch that we created earlier to check that it works as intended:

In [43]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


- The results above, where the first tensor represents the inputs, and the second tensor represents the targets.

- Let's see what this replacement by -100 accomplishes
  
- For illustration purposes, let's assume we have a small classification task with 2 class labels, 0 and 1, similar to chapter 6
  
- If we have the following logits values (outputs of the last layer of the model), we calculate the following loss

In [44]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


- Adding an additional token ID will, as we would expect, affect the loss calculation.

In [45]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


- Let's see what happens if we replace the class label of one of the examples with -100

In [46]:
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


- As we can see, the resulting loss on these 3 training examples is the same as the loss we calculated from the 2 training examples, which means that the cross-entropy loss function ignored the training example with the -100 label

- By default, PyTorch has the `cross_entropy(..., ignore_index=-100)` setting to ignore examples corresponding to the label `-100`

- Using this `-100 ignore_index`, we can ignore the additional end-of-text (padding) tokens in the batches that we used to pad the training examples to equal length

- However, we don't want to ignore the first instance of the end-of-text (padding) token (50256) because it can help signal to the LLM when the response is complete

- In practice, it is also common to mask out the target token IDs that correspond to the instruction, as illustrated in the figure below (this is a recommended reader exercise after completing the chapter)

![Alt text](../../assests/figure713.png)


- From the figure above: The left side shows the formatted input text we tokenize and then feed to the LLM during training. The right side shows the target text we prepare for the LLM where we can optionally mask out the instruction section, which means replacing the corresponding token IDs with the `-100` ignore
index value.


- By masking out the target token IDs that correspond to the instruction, as shown in figure above, the LLM `cross entropy` loss is only computed for the generated response target IDs. By masking out the instruction tokens, the model is trained to focus on generating accurate responses rather than additionally also memorizing instructions, which can help with reducing overfitting.

### **4. Creating data loaders for an instruction dataset**

- In this section, we use the `InstructionDataset` class and `custom_collate_fn` function to instantiate the training, validation, and test data loaders.

![Alt text](../../assests/figure714.png)

- In previous sections, we prepared the dataset and implemented a custom collate function for batching the instruction dataset. In this section, we create and apply the data loaders to the `training`, `validation`, and `test` sets that we need for the LLM instruction finetuning and evaluation.

- Another additional detail of the previous `custom_collate_fn` function is that we now directly move the data to the target device (e.g., GPU) instead of doing it in the main training loop, which improves efficiency because it can be carried out as a background process when we use the `custom_collate_fn` as part of the data loader.

- Using the `partial` function from Python's `functools` standard library, we create a new function with the device argument of the original function pre-filled.

In [47]:
if torch.cuda.is_available():
   device = torch.device("cuda")
elif torch.backends.mps.is_available():
   device = torch.device("mps")
else:
   device = torch.device("cpu")

print("Device:", device)

Device: mps


- Next, to reuse the chosen device setting in `custom_collate_fn` when we plug it into the `PyTorch DataLoader` class later in this section, we use the partial function from Python's functools standard library to create a new version of the function with the device argument pre-filled. 

- Additionally, we set the `allowed_max_length` to 1024, which truncates the data to the maximum context length supported by the GPT-2 model we finetune later.

In [48]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

- Next, we instantiate the data loaders similar to previous chapters, except that we now provide our own collate function for the batching process

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [50]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

- Let's see what the dimensions of the resulting input and target batches look like

In [51]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.

- As we can see based on the output above, all batches have a batch size of 8 but a different length, as expected

- Let's also double-check that the `inputs` contain the `<|endoftext|>` padding tokens corresponding to token ID `50256` by printing the contents of the first training example in the `inputs` batch

In [59]:
print(inputs[0])

tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,
          985,   576,    13,   198,   198, 21017, 23412,    25,   198,   464,
         5156,   318,   845, 13779,    13,   198,   198, 21017, 18261,    25,
          198,   464,  5156,   318,   355, 13779,   355,   257,  4936,    13,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
       device='mps:0')


- Similarly, we visually double-check that the `targets` contain the `-100` placeholder tokens

In [63]:
print(targets[0])

tensor([  318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,   257,
         2882,   326, 20431, 32543,   262,  2581,    13,   198,   198, 21017,
        46486,    25,   198, 30003,  6525,   262,  6827,  1262,   257,   985,
          576,    13,   198,   198, 21017, 23412,    25,   198,   464,  5156,
          318,   845, 13779,    13,   198,   198, 21017, 18261,    25,   198,
          464,  5156,   318,   355, 13779,   355,   257,  4936,    13, 50256,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
       device='mps:0')


### **5. Loading a pretrained LLM**

- In the previous sections, we spent a lot of time preparing the dataset for `instruction finetuning`, which is a key aspect of the supervised finetuning process. Many other aspects are the same as in pretraining, allowing us to reuse much of the code from earlier chapters

- Before beginning instruction finetuning, we first load a pretrained GPT model that we want to finetune.

![Alt text](../../assests/figure715.png)


- After the dataset preparation, the process of finetuning an LLM for instruction-following begins with loading a pretrained LLM, which serves as the foundation for subsequent training. This pretrained model, having already learned general language patterns and knowledge from vast amounts of text data, is then adapted for instruction following through the finetuning process in the next section.

- However, instead of loading the smallest `124` million parameter model, we load the medium version with `355` million parameters since the `124` million model is too small for achieving qualitatively reasonable results via instruction finetuning.

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt


BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# if you're using GPU
CHOOSE_MODEL = "gpt2-medium (355M)"

# For CPU and MPS
# CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

: 

- Let's take a moment to assess the pretrained LLM's performance on one of the validation tasks by comparing its output to the expected response.

- This will give us a baseline understanding of how well the model performs on an instruction-following task right out of the box, prior to finetuning, and will help us appreciate the impact of finetuning later on.

- Before we start finetuning the model in the next section, let's see how it performs on one of the validation tasks

In [ ]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

- Below is an instruction that describes a task. Write a response that appropriately completes the request.

- The content of the instruction are as follows:

```md
### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
```

```md
Next, we generate the model's response using the generate function from chapter 5:
```

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

- It's important to note that the `generate` function returns the combined input and output text. This behavior was convenient in previous chapters since pretrained LLMs are primarily designed as text-completion models, where the input and output are concatenated to create a coherent and legible text.

- To isolate the response, we can subtract the length of the instruction from the start of the `generated_text`.

In [ ]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

- As we can see, the model is not capable of following the instructions, yet; it creates a `"Response"` section but it simply repeats the original input sentence as well as the instruction, failing to convert the active sentence to passive voice as requested.

- In the next section, we implement the finetuning process to improve the model's ability to comprehend and appropriately respond to such requests.

### **6. Finetuning the LLM on instruction data**

- This section focuses on finetuning the LLM. 

- We take the pretrained model loaded in the previous section and further train it using the instruction dataset prepared earlier in this chapter.


![Alt text](../../assests/figure716.png)


- For the finetuning process itself, we can reuse the loss calculation and training functions implemented in the previous chapters during the pretraining:

In [ ]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

- Let's calculate the initial training and validation set loss before we start training (as in previous chapters, the goal is to minimize the loss)

In [ ]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

**DEALING WITH HARDWARE LIMITATIONS**

Note that using and training a larger model like `GPT-2` medium (355 million parameters) is more computationally intensive compared to the smaller `GPT-2` model (124 million parameter) used in the previous chapters. If you encounter issues due to hardware limitations, you can switch to the smaller model by changing `CHOOSE_MODEL = "gpt2-medium (355M)"` to `CHOOSE_MODEL = "gpt2-small (124M)"`. Alternatively, to speed up the model training, consider using a `GPU`. 

- The table below provides reference runtimes for training each model on various devices, including CPUs and GPUs. Running this code on a compatible GPU requires no code changes and can significantly speed up training.

![Alt text](../../assests/table1.png)


- With the model and data loaders prepared, we can now proceed to train the model. The following code sets up the training process, including initializing the optimizer, setting the number of epochs, and defining the evaluation frequency and starting context to evaluate generated LLm responses during training based on the first validation set instruction we looked at earlier:

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

- The following output above displays the training progress over two epochs, where a steady decrease in losses indicates improving ability to follow instructions and generate appropriate responses.

- The training output shows that the model is learning effectively, as we can tell based on the consistently decreasing training and validation loss values over the two epochs. This suggests that the model is gradually improving its ability to understand and follow the provided instructions. 

- Since the model demonstrated effective learning within these two epochs, extending the training to a third epcoh or more is not essential, and may even be counterproductive here as it could lead to increased overfitting.

- Moreover, the generated responses at the end of each epoch let us inspect the model's progress in correctly executing the given task in the validation set example.

- In this case, the model successfully converts the active sentence `"The chef cooks the meal every day."` into its passive voice counterpart: `"The meal is cooked every day by the chef."`

- Finally, let's take a look at the training and validation loss curves

In [ ]:
from previous_chapters import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

- The plot above is showing the training and validation loss trends over two epochs. The solid line represents the training loss, showing a sharp decrease before stabilizing, while the dotted line represents the validation loss, which follows a similar pattern.

- The model's performance on both the training and validation sets improves substantially over the course of training. The rapid decrease in losses during the initial phase indicates that the model is quickly learning meaningful patterns and representations from the data.

- As the training progresses to the second epoch, the losses continue to decrease but at a slower rate, suggesting that the model is finetuning its learned representations and converging to a stable solution.

- As we can see, the loss decreases sharply at the beginning of the first epoch, which means the model starts learning quickly.

- We can see that slight overfitting sets in at around 1 training epoch

### **7. Extracting and saving responses**

- In this section, we save the test set responses for scoring in the next section.

- We also save a copy of the model for future use.

- We proceed to evaluate the model performance on the held-out test set. To accomplish this, we first extract the model-generated responses for each input in the test dataset and collect them for manual analysis.

- But first, let's take a brief look at the responses generated by the finetuned model


![Alt text](../../assests/figure718.png)



In [ ]:
torch.manual_seed(123)

for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

- The `generate` function returns the combined input and output text, so we use slicing and the `.replace()` method on the `generated_text` contents to extract the model's response. 

- As we can see based on the test set instructions, given responses, and the model's responses, the model performs relatively well.

- The answers to the first and last instructions are clearly correct

- The second answer is close; the model answers with `"cumulus cloud"` instead of `"cumulonimbus"` (however, note that cumulus clouds can develop into cumulonimbus clouds, which are capable of producing thunderstorms)

- Most importantly, we can see that model evaluation is not as straightforward as in the previous chapter, where we just had to calculate the percentage of correct spam/non-spam class labels to obtain the classification accuracy

- In practice, instruction-finetuned LLMs such as chatbots are evaluated via multiple approaches
  - short-answer and multiple choice benchmarks such as MMLU ("Measuring Massive Multitask Language Understanding", https://arxiv.org/abs/2009.03300), which test the knowledge of a model

  - human preference comparison to other LLMs, such as LMSYS chatbot arena (https://arena.lmsys.org)
  
  - automated conversational benchmarks, where another LLM like GPT-4 is used to evaluate the responses, such as AlpacaEval (https://tatsu-lab.github.io/alpaca_eval/)

- In practice, it can be useful to consider all three types of evaluation methods: multiple choice question answering, human evaluation, and automated metrics that measure conversational performance. 

- Since we are interested in assessing conversational performance in this chapter rather than just the ability to answer multiple-choice questions, method 2 (`human evaluation`) and 3 (`automated metrics`) may be more relevant.

- Human evaluation, while providing valuable insights, can be relatively laborious and time-consuming, especially when dealing with a large number of responses. For instance, reading and assigning ratings to all 1,100 responses would require a significant amount of effort.

- So, considering the scale of the task at hand, we will implement an approach similar to `method 3`, which involves evaluating the responses automatically using another LLM. This will allow us to efficiently assess the quality of the generated responses without the need for extensive human involvement, thereby saving time and resources while still obtaining meaningful performance indicators.

- In the next section, we will use an approach similar to AlpacaEval and use another LLM to evaluate the responses of our model; however, we will use our own test set instead of using a publicly available benchmark dataset. This allows for a more targeted and relevant assessment of the model's performance within the context of our intended use cases represented in our instruction dataset.

- For this, we add the model response to the `test_data` dictionary and save it as a `"instruction-data-with-response.json"` file for record-keeping so that we can load and analyze it in separate Python sessions if needed

- The following code uses the `generate` method in the same manner as before; however, we now iterate over the entire `test_set`. Also, instead of printing the model responses, we add them to the `test_set` dictionary:

In [ ]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    test_data[i]["model_response"] = response_text


with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # "indent" for pretty-printing

- Let's double-check one of the entries to see whether the responses have been added to the `test_data` dictionary correctly

In [ ]:
print(test_data[0])

- Finally, we also save the model in case we want to reuse it in the future

In [ ]:
import re


file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"Model saved as {file_name}")

# Load model via
# model.load_state_dict(torch.load("gpt2-medium355M-sft.pth"))

### **8. Evaluating the finetuned LLM**

- Previously, we judged the performance of an instruction finetuned model by looking at its responses on `3` examples of the test set. While this gives us a rough idea of how well the model performs, this method does not really scale well to larger amounts of responses. 

- In this section, we implement a method to automate the response evaluation of the finetuned LLM using another, larger LLM.

![Alt text](../../assests/figure719.png)

- In this section, we automate the response evaluation of the finetuned LLM using another, larger LLM

- In particular, we use an instruction-finetuned 8-billion-parameter Llama 3 model by Meta AI that can be run locally via ollama (https://ollama.com)

- (Alternatively, if you prefer using a more capable LLM like GPT-4 via the OpenAI API, please see the [llm-instruction-eval-openai.ipynb notebook](https://github.com/rasbt/LLMs-from-scratch/blob/42c130623b28ec260b012f5accc61264e72dbb89/ch07/03_model-evaluation/llm-instruction-eval-openai.ipynb))


- Ollama is an application to run LLMs efficiently on your laptop.

- It is a wrapper around llama.cpp (https://github.com/ggerganov/llama.cpp), which implements LLMs in pure C/C++ to maximize efficiency

- Note that it is a tool for using LLMs to generate text (inference), not training or finetuning LLMs

- Before running the code below, install ollama by visiting https://ollama.com and following the instructions (for instance, clicking on the "Download" button and downloading the ollama application for your operating system)

- For macOS and Windows users, click on the ollama application you downloaded; if it prompts you to install the command line usage, say "yes"

- Linux users can use the installation command provided on the ollama website

- In general, before we can use ollama from the command line, we have to either start the ollama application or run `ollama serve` in a separate terminal


![Alt text](../../assests/figure720.png)


- With the ollama application or `ollama serve` running in a different terminal, on the command line, execute the following command to try out the 8-billion-parameter Llama 3 model (the model, which takes up 4.7 GB of storage space, will be automatically downloaded the first time you execute this command).

`# 8B model`
```sh
ollama run llama3
```
The output looks like as follows


```sh
$ ollama run llama3
pulling manifest
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- Note that `llama3` refers to the instruction finetuned 8-billion-parameter Llama 3 model

- Using ollama with the `"llama3"` model (a 8B parameter model) requires 16 GB of RAM; if this is not supported by your machine, you can try the smaller model, such as the 3.8B parameter phi-3 model by setting `model = "phi-3"`, which only requires 8 GB of RAM

- Alternatively, you can also use the larger 70-billion-parameter Llama 3 model, if your machine supports it, by replacing `llama3` with `llama3:70b`

- After the download has been completed, you will see a command line prompt that allows you to chat with the model

- Try a prompt like "What do llamas eat?", which should return an output similar to the following


```sh
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered
stomach and eat plants that are high in fiber. In the wild, llamas
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall
grasses, wheat, oats, and barley.
```

- You can end this session using the input `/bye`

- The following code checks whether the `ollama` session is running correctly before proceeding to use `ollama` to evaluate the test set responses we generated in the previous section

In [4]:
import psutil

In [5]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [ ]:
# This cell is optional; it allows you to restart the notebook
# and only run section 7.7(previous section) without rerunning any of the previous code
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- Now, an alternative way to the `ollama run` command we used earlier to interact with the model is via its REST API in Python via the following function

- Before you run the next cells in this notebook, make sure that ollama is still running (the previous code cells should print `"Ollama running: True"`)

- Next, run the following code cell to query the model

In [8]:
import urllib.request
import json

def query_model(
    prompt,
    model="llama3",
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }


    # Convert the dictionary to a JSON formatted string and encode it to bytes
    payload = json.dumps(data).encode("utf-8")

    # Create a request object, setting the method to POST and adding necessary headers
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )
    request.add_header("Content-Type", "application/json")

    # Send the request and capture the response
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # Read and decode the response
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data


# model = "llama3"
model = "llama2"
result = query_model("What do Llamas eat?", model)
print(result)


Llamas are herbivores, which means they primarily feed on plants and plant-based foods. Their diet typically consists of:

1. Grasses: Llamas can eat a variety of grasses, including timothy grass, alfalfa, and clover.
2. Hay: Llamas can eat hay, which is dried and stored grasses or legumes.
3. Grains: Llamas can eat grains such as oats, barley, and corn.
4. Vegetables: Llamas can eat a variety of vegetables, including leafy greens like kale and spinach, as well as root vegetables like carrots and sweet potatoes.
5. Fruits: Llamas can eat fruits like apples, bananas, and berries.
6. Leaves: Llamas can eat leaves from trees and shrubs, such as willow and cottonwood.
7. Bark: Llamas can eat the bark of certain trees, such as willow and cottonwood.
8. Twigs: Llamas can eat twigs and small branches.
9. Flowers: Llamas can eat flowers, including wildflowers like daisies and marigolds.
10. Salt: Llamas need salt to stay healthy, so they will often seek out salt licks or salt blocks in their 

- Now, using the `query_model` function we defined above, we can evaluate the responses of our finetuned model; let's try it out on the first 3 test set responses we looked at in a previous section

In [26]:
test_data[:3]

[{'instruction': 'Rewrite the sentence using a simile.',
  'input': 'The car is very fast.',
  'output': 'The car is as fast as lightning.'},
 {'instruction': 'What type of cloud is typically associated with thunderstorms?',
  'input': '',
  'output': 'The type of cloud typically associated with thunderstorms is cumulonimbus.'},
 {'instruction': "Name the author of 'Pride and Prejudice'.",
  'input': '',
  'output': 'Jane Austen.'}]

In [30]:
for entry in test_data[:3]:
    print(entry)

{'instruction': 'Rewrite the sentence using a simile.', 'input': 'The car is very fast.', 'output': 'The car is as fast as lightning.'}
{'instruction': 'What type of cloud is typically associated with thunderstorms?', 'input': '', 'output': 'The type of cloud typically associated with thunderstorms is cumulonimbus.'}
{'instruction': "Name the author of 'Pride and Prejudice'.", 'input': '', 'output': 'Jane Austen.'}


In [ ]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")

---
**Note: Better evaluation prompt**

- [A reader (Ayoosh Kathuria)](https://github.com/rasbt/LLMs-from-scratch/discussions/449) suggested a longer, improved prompt that evaluates responses on a scale of 1–5 (instead of 1 to 100) and employs a grading rubric, resulting in more accurate and less noisy evaluations:


prompt = """
You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance.
You will be given an instruction, a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing the evaluation criteria.
Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
Please do not generate any other opening, closing, and explanations.


**Here is the rubric you should use to build your answer:**

1: The response fails to address the instructions, providing irrelevant, incorrect, or excessively verbose information that detracts from the user's request.

2: The response partially addresses the instructions but includes significant inaccuracies, irrelevant details, or excessive elaboration that detracts from the main task.

3: The response follows the instructions with some minor inaccuracies or omissions. It is generally relevant and clear, but may include some unnecessary details or could be more concise.

4: The response adheres to the instructions, offering clear, accurate, and relevant information in a concise manner, with only occasional, minor instances of excessive detail or slight lack of clarity.

5: The response fully adheres to the instructions, providing a clear, accurate, and relevant answer in a concise and efficient manner. It addresses all aspects of the request without unnecessary details or elaboration

**Provide your feedback as follows:**

Feedback:::

Evaluation: (your rationale for the rating, as a text)

Total rating: (your rating, as a number between 1 and 5)


You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the instruction, the reference answer, and the response.

Instruction: {instruction}

Reference Answer: {reference}

Answer: {answer}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.

Feedback:::

Evaluation: """

---

- Based on the generated responses, we can observe that the `Llama 3` model provides reasonable evaluations and is capable of assigning partial points when a model's answer is not entirely correct. For instance, if we consider the evaluation of the `"cumulus cloud"` answer, the model acknowledges the partial correctness of the response.

- The previous prompt returns highly detailed evaluations in addition to the score. We can modify the prompt to just generate integer scores ranging from `0` to `100`, where `100` represents the best possible score. This modification allows us to calculate an average score for our model, which serves as a more concise and quantitative assessment of its performance.


- The evaluation of the 110 entries in the test set takes about 1 minute on an M3 MacBook Air laptop

- The following `generate_model_scores` function uses a modified prompt telling the model to `"Respond with the integer number only"`:

In [ ]:
def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

- The evaluation output shows that our finetuned model achieves an average score above 50, which provides a useful benchmark for comparison aganist other models or for experimenting with different training configurations to improve the model's performance.

- Note that ollama is not fully deterministic across operating systems (as of this writing), so the numbers you are getting might slightly differ from the ones shown above


- To further improve our model's performance, we can explore various strategies, such as:
  
  - Adjusting the hyperparameters during finetuning, such as the learning rate, batch size, or number of epochs.
  
  - Increasing the size of the training dataset or diversifying the examples to cover a broader range of topics and styles.
  
  - Experimenting with different prompts or instruction formats to guide the model's responses more effectively.

  - Considering the use of a larger pretrained model, which may have greater capacity to capture complex patterns and generate more accurate responses.

- For reference, the original

  - Llama 3 8B base model achieves a score of 58.51

  - Llama 3 8B instruct model achieves a score of 82.65

### **8. Conclusions**

- We covered the major steps of the LLM development cycle: implementing an LLM architecture, pretraining an LLM, and finetuning it


![Alt text](../../assests/figure721.png)


- An overview of the different stages of implementing, pretraining, and finetuning an LLM.

**What's next?**

- An optional step that is sometimes followed after instruction finetuning, as described in this chapter, is `preference finetuning`.

- `Preference finetuning` is particularly useful for customizing a model to better align with specific user preferences.

**Staying up to date in a fast-moving field**

The fields of AI and LLM research are evolving at a rapid (and depending on who you ask, exciting) pace. One way to keep up with the latest advancements, consider exploring recent research papers on arXiv at https://arxiv.org/list/cs.LG/recent.